# Features

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)

df = pd.read_csv(Path("../data/raw/sleep_health.csv"))
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## Cleaning

In [2]:
df["BMI Category"] = df["BMI Category"].replace({"Normal Weight": "Normal"})
df["BMI Category"].value_counts()

BMI Category
Normal        216
Overweight    148
Obese          10
Name: count, dtype: int64

In [3]:
bp = df["Blood Pressure"].str.split("/", expand=True).astype(int)
df["Systolic"] = bp[0]
df["Diastolic"] = bp[1]
df = df.drop(columns=["Blood Pressure"])
df[["Systolic", "Diastolic"]].head()

,Systolic,Diastolic
0,126,83
1,125,80
2,125,80
3,140,90
4,140,90


In [4]:
df["Sleep Disorder"] = df["Sleep Disorder"].fillna("No disorder")
df["Sleep Disorder"].value_counts()

Sleep Disorder
No disorder    219
Sleep Apnea     78
Insomnia        77
Name: count, dtype: int64

## What goes into the model

Out: `Person ID` (identifier). `Sleep Disorder`, kept aside to check the clusters. `Occupation`: eleven levels, four of them under five people, and as eleven yes/no columns it takes over the partition. `Gender`: in an app the question is optional and not everyone answers male or female, so the model should not need it. Both are looked at in `analysis/columns.ipynb`.

In: everything else, including heart rate, blood pressure, BMI and age. The profile is meant to cover the person, habits and physiology together. Age is always known and dropping it changes almost nothing, see the same notebook.

In [5]:
df["BMI"] = df["BMI Category"].map({"Normal": 0, "Overweight": 1, "Obese": 2})

candidates = [
    "Age",
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "Heart Rate",
    "Daily Steps",
    "Systolic",
    "Diastolic",
    "BMI",
]
df[candidates].corr().round(2)

,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps,Systolic,Diastolic,BMI
Age,1.00,0.34,0.47,0.18,-0.42,-0.23,0.06,0.61,0.59,0.43
Sleep Duration,0.34,1.00,0.88,0.21,-0.81,-0.52,-0.04,-0.18,-0.17,-0.35
Quality of Sleep,0.47,0.88,1.00,0.19,-0.90,-0.66,0.02,-0.12,-0.11,-0.32
Physical Activity Level,0.18,0.21,0.19,1.00,-0.03,0.14,0.77,0.27,0.38,0.05
Stress Level,-0.42,-0.81,-0.90,-0.03,1.00,0.67,0.19,0.10,0.09,0.16
Heart Rate,-0.23,-0.52,-0.66,0.14,0.67,1.00,-0.03,0.29,0.27,0.47
Daily Steps,0.06,-0.04,0.02,0.77,0.19,-0.03,1.00,0.10,0.24,-0.14
Systolic,0.61,-0.18,-0.12,0.27,0.10,0.29,0.10,1.00,0.97,0.74
Diastolic,0.59,-0.17,-0.11,0.38,0.09,0.27,0.24,0.97,1.00,0.74
BMI,0.43,-0.35,-0.32,0.05,0.16,0.47,-0.14,0.74,0.74,1.00


`BMI Category` has an order, Normal < Overweight < Obese, so it becomes 0, 1, 2 rather than three yes/no columns. One number keeps the order and takes one dimension instead of three. It does assume the two steps are the same size, which is a simplification.

`Underweight` does not appear in the data. A user in that category is outside what the model has seen, and the code rejects the row with an error.

Three pairs are strongly correlated: systolic and diastolic (0.97), sleep duration and quality (0.88), activity and steps (0.77). Sleep duration, quality and stress also move together, between 0.8 and 0.9; I leave it as it is.

## Redundant pairs

Two of the three pairs measure the same thing twice and become a single column. The third does not, and stays as two.

In [6]:
df["Mean Arterial Pressure"] = df["Diastolic"] + (df["Systolic"] - df["Diastolic"]) / 3
df[["Systolic", "Diastolic", "Mean Arterial Pressure"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Systolic,374.0,128.553476,7.748118,115.000000,125.0,130.0,135.0,142.0
Diastolic,374.0,84.649733,6.161611,75.000000,80.0,85.0,90.0,95.0
Mean Arterial Pressure,374.0,99.284314,6.647311,88.333333,95.0,100.0,105.0,110.0


Mean arterial pressure is the usual way to turn a blood pressure reading into one number.

Sleep duration and quality of sleep correlate at 0.88, but hours and a 1 to 10 score answer two different questions: how much you sleep and how well. Someone who sleeps eight hours badly and someone who sleeps six hours well would get the same average. How many people are in one of those two situations, one column above its median and the other below:

In [7]:
above_median_hours = df["Sleep Duration"] > df["Sleep Duration"].median()
above_median_quality = df["Quality of Sleep"] > df["Quality of Sleep"].median()
(above_median_hours != above_median_quality).sum()

np.int64(88)

Almost a quarter of the people. The two columns stay separate. The 0.88 is also a feature of this synthetic data: in real life hours and quality go together much less.

In [8]:
activity = df["Physical Activity Level"]
steps = df["Daily Steps"]
df["Activity Index"] = (
    (activity - activity.mean()) / activity.std() + (steps - steps.mean()) / steps.std()
) / 2
df["Activity Index"].describe().round(2)

count    374.00
mean      -0.00
std        0.94
min       -1.88
25%       -0.59
50%        0.08
75%        0.75
max        1.72
Name: Activity Index, dtype: float64

Minutes and steps are on different scales, so both are put on the same scale first and then averaged.

Stress and heart rate are correlated with each other and with sleep, but one is self reported and the other is measured. They stay as they are.

Four of the eight columns are about sleep and stress, so distances count that side more than activity. I keep it this way so the columns stay readable, and for the same reason I do not use PCA: the profiles have to be read by a person and by a language model.

## Save

In [9]:
out = Path("../data/processed")
out.mkdir(exist_ok=True)
df.to_csv(out / "sleep_health.csv", index=False)
df.shape

(374, 17)

## Same thing from the package

`advisor.features.WellnessFeatures` does the steps above in code, so the model and a new user go through the same transformation. Two checks: that it gives the same numbers as this notebook on the full data, and that a row transformed on its own, with the statistics learned on other rows, comes out the same as in a batch.

In [10]:
from advisor.features import FEATURES, WellnessFeatures

raw = pd.read_csv(Path("../data/raw/sleep_health.csv"))
from_package = WellnessFeatures().fit_transform(raw)
print("max difference with this notebook:", (from_package - df[FEATURES]).abs().max().max())

fitted = WellnessFeatures().fit(raw.iloc[:300])
one_row = fitted.transform(raw.iloc[[350]])
batch = fitted.transform(raw.iloc[300:])
print(
    "max difference, one row alone vs in a batch:", (one_row.iloc[0] - batch.loc[350]).abs().max()
)
one_row.round(3)

max difference with this notebook: 0.0
max difference, one row alone vs in a batch: 0.0


,Age,Sleep Duration,Quality of Sleep,Activity Index,Stress Level,Heart Rate,Mean Arterial Pressure,BMI
350,57,8.1,9,0.384,3,68,110.0,1


The activity index of a single row is not zero, so `transform` uses the mean and spread learned in `fit` rather than recomputing them on the row.